# Coding-docs RAG poisoning - Week 2 TEST notebook

**Runtime: CPU (High-RAM if you have it). Do NOT mount Drive. Do NOT add API keys.**
Network stays on (some BigCodeBench tests use it). This machine only runs code and
gives back pass/fail.

Order: setup -> build environments -> (new session only: restore) -> DS-1000 Codex check ->
reference solutions, one dataset per cell -> upload answers -> score them.

Everything in this session (environments and `/content/test_outputs`) is lost when the
session ends. So every check cell ends by downloading `test_outputs.zip` with **all**
results so far. Keep the newest zip. In a new session: setup, build environments, then
the restore cell with that zip, and continue where you stopped.

In [ ]:
REPO_URL = "https://github.com/MrunalKotkar/Code-Rag-Defense"
import os, shutil, subprocess
from google.colab import files
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "--ff-only"], check=True)
os.chdir("/content/repo")
OUTPUTS = "/content/test_outputs"

def save_outputs():
    """Download everything in /content/test_outputs as one zip (all results so far)."""
    if not os.path.isdir(OUTPUTS) or not os.listdir(OUTPUTS):
        print("nothing to save yet")
        return
    shutil.make_archive("/content/test_outputs", "zip", OUTPUTS)
    print("saving:", sorted(os.listdir(OUTPUTS)))
    files.download("/content/test_outputs.zip")

print("ready")

## Day 1-2 - Build the three environments (Python 3.10 each; takes a while)

In [ ]:
!python -m coding_track.testing.build_envs ds1000 bcb odex

## New session only - restore the newest `test_outputs.zip` you downloaded
Skip this in the first session.

In [ ]:
uploaded = list(files.upload())
shutil.unpack_archive(uploaded[0], OUTPUTS, 'zip')
print(sorted(os.listdir(OUTPUTS)))

## Day 1 - DS-1000 check: reproduce Codex-002 (expected overall 0.388)

In [ ]:
!python -m coding_track.testing.run_tests codex
save_outputs()

## Day 2 - Official reference solutions, one dataset per cell
Each cell adds its dataset to `reference_results.jsonl` (the others are kept) and downloads the zip.

In [ ]:
!python -m coding_track.testing.run_tests reference --datasets ds1000
save_outputs()

In [ ]:
!python -m coding_track.testing.run_tests reference --datasets odex
save_outputs()

In [ ]:
!python -m coding_track.testing.run_tests reference --datasets bigcodebench
save_outputs()

## Pilot / full scoring - upload the answers file exported by the main notebook

In [ ]:
uploaded = list(files.upload())
print(uploaded)

In [ ]:
!python -m coding_track.testing.run_tests answers --input {uploaded[0]}
save_outputs()